# CSE476 - Multi-Agent Customer Support Demo

**Project:** Enterprise Customer Support Triage and Resolution Desk
**Stack:** Semantic Kernel · Groq (free tier) · Supabase PostgreSQL + pgvector · Sentence-Transformers · FastAPI · Streamlit

This notebook demonstrates the complete runnable workflow:
classifier -> specialist (billing / technical / account) -> RAG -> escalation -> human verdict.

> Requirements: `.env` with `GROQ_API_KEY` and `DATABASE_URL`, migrations applied,
> and `python run.py ingest` executed once (or run the ingestion cell below).


## Architecture (router-to-specialists)

```mermaid
flowchart LR
  C[Customer] --> CLS[Classifier Agent]
  CLS -->|billing| B[Billing Specialist]
  CLS -->|technical| T[Technical Specialist]
  CLS -->|account| A[Account Specialist]
  CLS -->|low confidence / other| E[Escalation Router]
  B --> R{Resolved?}
  T --> R
  A --> R
  R -->|no| E
  E --> H[Human Support]
  R -->|yes| DONE[Final resolution]
  T <--> V[(pgvector knowledge)]
  CLS & B & T & A & E -. logs .-> L[(Supabase agent_logs)]
```


In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from llm.model_config import validate_configuration
from database.connection import check_database_connection

print("Configuration:", json.dumps(validate_configuration(), indent=2))
DB_OK, DB_MESSAGE = check_database_connection()
print("Database:", DB_MESSAGE)


## Step 1-3 - Initialise the system, Supabase connection and Groq client

All agents share one Groq client and one Semantic Kernel instance. The
coordinator owns routing, shared ticket state and agent handoffs.


In [ ]:
from database.repository import Repository
from llm.groq_client import GroqClient
from memory.conversation_memory import ConversationMemory
from rag.retriever import KnowledgeRetriever
from orchestration.coordinator import SupportCoordinator

repository = Repository()
groq = GroqClient()
memory = ConversationMemory(repository)
retriever = KnowledgeRetriever(repository)
coordinator = SupportCoordinator(
    repository=repository, groq=groq, memory=memory, retriever=retriever
)

print("Groq configured:", groq.available, "| model:", groq.model)
print("Agents:")
for agent in coordinator.get_agent_roster():
    print(f"  - {agent['name']}: tools={agent['tools']}")


## Step 4-5 - Load knowledge documents and generate local embeddings

Chunking uses overlapping word windows; embeddings are 384-dim (`all-MiniLM-L6-v2`).

In [ ]:
from rag.chunking import chunk_text
from rag.document_loader import load_directory

documents = load_directory(ROOT / "knowledge_base")
print(f"Loaded {len(documents)} documents")
for doc in documents:
    chunks = chunk_text(doc["content"])
    print(f"  {doc['title']:26s} category={doc['category']:9s} chunks={len(chunks)}")


## Step 5b - Store documents + embeddings in Supabase pgvector

The same PostgreSQL database stores tickets, logs, conversations, payments and vectors. No second database is used.

In [ ]:
from rag.ingestion import KnowledgeIngestion

if DB_OK:
    stats = KnowledgeIngestion(repository).ingest_directory()
    print(json.dumps(stats, indent=2))
else:
    print("SKIPPED: configure DATABASE_URL and run migrations, then rerun this cell.")


## Step 6 - Submit a customer complaint (saved in Supabase)

In [ ]:
ticket_id = None
if DB_OK:
    user = repository.get_or_create_user("Notebook Demo", "notebook.demo@example.com")
    repository.insert_payment_record(
        user_id=user["id"], transaction_ref="TXN1001",
        amount=49.99, status="completed", description="Pro subscription",
    )
    repository.insert_payment_record(
        user_id=user["id"], transaction_ref="TXN1002",
        amount=49.99, status="completed", description="Pro duplicate",
    )
    ticket = repository.create_ticket(
        user_id=user["id"],
        subject="My payment was deducted twice",
        message="I was charged twice for TXN1001. Please refund the duplicate.",
        priority="high",
    )
    ticket_id = ticket["id"]
    memory.add_customer_message(ticket_id, ticket["message"], user_id=user["id"])
    print("Ticket created:", ticket_id)
else:
    print("SKIPPED: database not configured.")


## Step 7 - Run the Classifier Agent (tools + Groq + validation)

In [ ]:
import asyncio
from orchestration.shared_state import SharedTicketState

if DB_OK and ticket_id:
    ticket = repository.get_ticket(ticket_id)
    state = SharedTicketState.from_ticket(ticket)
    classification = asyncio.run(coordinator.classifier.run(ticket, state))
    print(json.dumps(classification.to_dict(), indent=2))
else:
    print("SKIPPED: database not configured.")


## Step 8 - Routing decision executed through Semantic Kernel

In [ ]:
if DB_OK and ticket_id:
    category = classification.output.get("category", "other")
    confidence = classification.output.get("confidence", 0.0)
    route = asyncio.run(coordinator._route(category, confidence))
    print(json.dumps(route, indent=2))
else:
    print("SKIPPED")


## Step 9-11 - Invoke the Billing Specialist: tools, RAG, shared state, reply

The Billing Specialist calls `lookup_payment_record` and `check_billing_policy` through the Semantic Kernel plugin, then generates a policy-grounded reply.

In [ ]:
if DB_OK and ticket_id:
    final = coordinator.process_ticket_sync(ticket_id)
    print("Status         :", final.get("status"))
    print("Category       :", final.get("category"))
    print("Assigned agent :", final.get("assigned_agent"))
    print("Reply          :", final.get("customer_reply"))
    print("\nExecution trace:")
    print(json.dumps(final.get("trace", []), indent=2))
else:
    print("SKIPPED")


## Step 10b - Technical pipeline with real pgvector RAG

In [ ]:
technical_ticket_id = None
if DB_OK and groq.available:
    user = repository.get_or_create_user("Notebook Demo", "notebook.demo@example.com")
    technical_ticket = repository.create_ticket(
        user_id=user["id"],
        subject="App crashes on login",
        message="My application crashes whenever I try to log in on Android 14.",
    )
    technical_ticket_id = technical_ticket["id"]
    memory.add_customer_message(technical_ticket_id, technical_ticket["message"], user_id=user["id"])
    tech_final = coordinator.process_ticket_sync(technical_ticket_id)
    print("Status  :", tech_final.get("status"))
    print("Agent   :", tech_final.get("assigned_agent"))
    print("Sources :", tech_final.get("sources"))
    print("Reply   :", tech_final.get("customer_reply"))
else:
    print("SKIPPED: requires DATABASE_URL, GROQ_API_KEY and ingested knowledge base.")


## Step 12-13 - Escalation demo (low confidence -> human queue)

In [ ]:
escalation_ticket_id = None
escalation = None
if DB_OK and groq.available:
    user = repository.get_or_create_user("Notebook Demo", "notebook.demo@example.com")
    unclear = repository.create_ticket(
        user_id=user["id"],
        subject="Not sure",
        message="Something feels off with my stuff. Not sure what.",
    )
    escalation_ticket_id = unclear["id"]
    memory.add_customer_message(escalation_ticket_id, unclear["message"], user_id=user["id"])
    esc_final = coordinator.process_ticket_sync(escalation_ticket_id)
    print("Status         :", esc_final.get("status"))
    print("Confidence     :", esc_final.get("confidence"))
    print("Customer reply :", esc_final.get("customer_reply"))
    escalation = repository.get_escalation_by_ticket(escalation_ticket_id)
    print("\nEscalation record:")
    print(json.dumps(escalation, indent=2))
else:
    print("SKIPPED")


## Step 14 - Human-in-the-loop verdict resolves the ticket

In [ ]:
if DB_OK and escalation:
    outcome = coordinator.escalation.process_human_verdict(
        escalation_id=escalation["id"],
        verdict="Called the customer - it was a general question, answered manually.",
        resolution="Your question was answered by our support team. Ticket closed.",
        action="resolve",
        operator="Human Support",
    )
    print("Ticket status  :", outcome["ticket_status"])
    print("Escalation     :", outcome["escalation"]["status"])
else:
    print("SKIPPED")


## Step 15 - Append-only audit log (every agent + human action)

In [ ]:
if DB_OK and escalation_ticket_id:
    print("Supabase agent_logs for the escalation ticket:")
    for log in repository.list_agent_logs(ticket_id=escalation_ticket_id, limit=30):
        print(f"  [{log['created_at']}] {log['agent_name']:20s} {log['action']:28s} {log['execution_status']}")
    print("\nGroq usage this session:", json.dumps(groq.get_usage(), indent=2))
else:
    print("SKIPPED")


## Viva checklist

- **Five independent agents**, each its own module with >= 2 real tools.
- **Semantic Kernel** hosts the plugins and Groq chat service; tools execute via `kernel.invoke`.
- **Router-to-specialists** coordination with shared ticket state and real handoffs.
- **Supabase PostgreSQL only** - tickets, conversations, logs, payments, vectors.
- **pgvector RAG** - local 384-dim embeddings, cosine similarity SQL function.
- **Persistent shared memory** - follow-up questions retrieve full history.
- **Human-in-the-loop** - verdicts update escalations + tickets + audit log.
- **Failure handling** - rate limits preserve tickets; missing RAG context escalates to a human.

Run the automated suite with `python run.py test` (36 unit tests + opt-in live integration tests).
